# Textual Data Analysis Pipeline Dashboard

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 匯入重構後的處理管線
from utils.pipeline_preprocess import run_preprocessing
from utils.pipeline_features import build_features
from utils.pipeline_modeling import evaluate_all_models

### Step 1: Data Preprocessing (資料前處理)
從 `data/raw_json` 讀取資料，萃取內容，並清除空白記錄。

**💡 執行這段會產生什麼？**
1. **實體檔案**：
   - `artifacts/reports/judgment_labels.xlsx`: 自動整理好的全案件勝敗訴、以及您剛設定的三種和解標籤。
   - `data/IP_Law_cases/`: 把過濾後的智財案件原始 JSON 單獨複製到這。
   - `artifacts/reports/fact_removed_blank.xlsx`: 取代原本龐大判決的乾淨「犯罪事實」與「理由」特徵文字檔。
2. **記憶體回傳值變數 (`df_clean`)**：內含判決 JID 與乾淨字串的 DataFrame，以及下方會印出預覽表格。

In [6]:
df_clean = run_preprocessing(
    input_folder="../data/raw_json", 
    output_folder="../data/IP_Law_cases", 
    artifacts_folder="../artifacts/reports",
    n_jobs=-1  # n_jobs=1 為單純循序運算; n_jobs=-1 代表啟用平行運算使用所有資源
)
display(df_clean.head())

🚀 緩存不存在，啟動 Cold Start (讀取自 /Users/xinc./Documents/GitHub/Personal-Project/textual data analysis/data/raw_json)...


Extracting Raw Data (Unified): 100%|██████████| 90027/90027 [00:12<00:00, 7042.53it/s] 


📦 正在獨立存儲「全文本備份」資料 (JID, JFULL) -> /Users/xinc./Documents/GitHub/Personal-Project/textual data analysis/artifacts/cache/full_text_backup.parquet
⚡️ 正在存儲「輕量化核心特徵」快取 -> /Users/xinc./Documents/GitHub/Personal-Project/textual data analysis/artifacts/cache/raw_extracted.parquet
⚖️ 正在執行標籤判定 (Pattern Matching)...
📊 正在產出 Excel 報表...
  - 正在產出 judgment_labels.xlsx...
  - 正在產出 raw_extracted.xlsx...
✅ Excel 報表產出完成！
✨ 全流程耗時: 1.76 分鐘


,JID,JYEAR,JCASE,JDATE,JTITLE,IP Law,JTYPE,main_clause,plaintiff_names,plaintiff_is_company,...,is_admin,is_appeal,is_first_instance,is_summary_case,claim_damages,claim_injunction,claim_destroy_goods,claim_validity_review,claim_admin_cancellation,VERDICT
0,"TPBA,91,訴,2130,20030709,2",91,訴,20030709,商標撤銷,True,RULING,原告之訴駁回。訴訟費用由原告負擔。,[甲○○○○○○],False,...,False,False,True,False,False,False,False,False,False,Other
1,"TPBA,90,訴,5379,20021011,2",90,訴,20021011,商標異議,True,RULING,原告之訴駁回。訴訟費用由原告負擔。,[百仙製藥工業股份有限公司],True,...,False,False,True,False,False,False,False,False,False,Other
2,"TYDM,104,聲,1011,20150316,1",104,聲,20150316,聲請沒收,False,RULING,扣案之盜版遊戲光碟捌片均沒收。,[],False,...,False,False,True,False,False,False,True,False,False,Other
3,"TPHM,89,上易,4570,20001020,1",89,上易,20001020,違反著作權法,True,CRIMINAL,原判決撤銷。甲○○共同連續意圖銷售而擅自以重製之方法侵害他人之著作財產權，處有期徒刑拾月。扣...,[],False,...,False,True,False,False,False,False,True,True,False,Lose
4,"TCDM,88,訴,1933,20010105",88,訴,20010105,違反著作權法,True,CRIMINAL,甲○○以明知為侵害著作權之物而意圖營利而交付之方法侵害他人之著作權為常業，處有期徒刑壹年拾月...,[],False,...,False,False,True,False,False,False,True,False,False,Lose


### Step 2: Tokenization & DTM (斷詞與特徵矩陣)
使用 CKIP 建立斷詞，並計算 Bag-of-Words (BoW) 與 TF-IDF 矩陣。

**💡 執行這段會產生什麼？**
1. **實體檔案 (非常耗時，通常只需跑一次)**：
   - `artifacts/reports/word_seg.xlsx`: 儲存經過 CkipWordSegmenter 斷詞後的每個字串陣列。
   - `artifacts/reports/verdict_results.xlsx`: 對齊過斷詞順序的最終 One-Hot 標籤集。
   - `lexicon_resources/dtm_csr_BoW.npz`: Bag of Words 的 Sparse 特徵稀疏矩陣。
   - `lexicon_resources/dtm_csr_TF_IDF.npz`: TF-IDF 的 Sparse 特徵稀疏矩陣。
2. **記憶體回傳值變數 (`dtm_features`)**：回傳建立好的特徵矩陣以供查看維度。

In [ ]:
# 如果已經跑過並存至檔案，會自動讀取而跳過重複斷詞
dtm_features = build_features(
    df_clean_path="../artifacts/reports/fact_removed_blank.xlsx",
    artifacts_folder="../artifacts/reports",
    lexicon_folder="../lexicon_resources"
)

### Step 3: Modeling & Evaluation (模型訓練與評估)
呼叫 `algos/` 資料夾中的各種演算法進行驗證與測試。

**💡 執行這段會產生什麼？**
1. **終端/畫面輸出**：
   - 逐步顯示 Train/Test 的資料拆分大小。
   - MNIR 萃取特徵的轉換紀錄（`mnir_z_train_bow.npy`）。
   - 各個 Grid Search 超參數(如 SVM的 C 值)的尋優過程、最佳 F1-Mean 數值。
2. **實體檔案**：如果您後續擴充呼叫了 PyTorch 類神經網路，會另外存下最佳的 `.pt` 權重。
3. **記憶體回傳值變數 (`results_df`)**：回傳一個包含各模型 (SVM, NB, RF 等) 測試集 Accuracy 與 F1-Score 的 DataFrame，自動印出漂亮評估表供論文擷圖使用。

In [ ]:
results_df = evaluate_all_models(
    dtm_bow_path="../lexicon_resources/dtm_csr_BoW.npz",
    dtm_tfidf_path="../lexicon_resources/dtm_csr_TF_IDF.npz",
    verdict_results_path="../artifacts/reports/verdict_results.xlsx"
)

display(results_df)